# SLR Gemini
A tool to support selection activities in Systematic Literature Reviews using ChatGPT

*By: [GitHub]()

[...]

## 1. Configure the environment and installing the necessary dependencies to run the tool. (no intervention is required in this session)

In [ ]:
# @title Imports {display-mode: "form"}


!pip install openai
from openai import OpenAI
from google.colab import files
from datetime import datetime
import os
import json
import numpy as np
import pandas as pd
import time

## 2. Insert the path of the json file containing title, summary and keywords for each paper, like:


```
[
    {
        "title": "title paper 1",
        "abstract": "abstract paper 1",
        "keywords":"keywords paper 1"
    },
    {
        "title": "title paper n",
        "abstract": "abstract paper n",
        "keywords":"keywords paper n"
    }
]
```

Tip: Use [this online tool](https://tableconvert.com/html-to-json) to generate json from table.

In [2]:
# @title Path JSON file
path = "data.json" # @param {type:"string"}
# path = 'data.json'

#read file
with open(path, 'r') as file:
  data = json.load(file)
#creat np to result
result_np = np.empty((len(data),9), dtype=object)

## 3. Get APIKey
In https://platform.openai.com/api-keys

You need to make a payment of at least USD $5 to release the gpt-4 model.

In [3]:
# @title Enter OpenAI APIKey
apiKey = "" # @param {type:"string"}
os.environ["OPENAI_API_KEY"] = apiKey

## 4. Select gpt model, see more in https://platform.openai.com/docs/models/

In [ ]:
# @title Enter the Gemini model you want to use
modelGemini = "gemini-2.5-flash"  # @param ["gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.0-flash"]

# Install the new SDK (run once)
# !pip install google-genai

from google import genai
from google.genai import types

# processing
i = 0

# Context
client = genai.Client(api_key="YOUR_GEMINI_API_KEY")  # or set env var GEMINI_API_KEY
print("Starting...")
print("Using Model " + modelGemini)

for item in data:
    startTime = time.perf_counter()
    result_np[i, 0] = item["title"]

    # System instruction shared across all calls
    system_instruction = (
        "Assume you are a software engineering researcher. Conducting a systematic literature review (SLR). "
        "Consider the title, abstract and keywords of a primary study.\n\n"
        f"Title: {item['title']}\n\n"
        f"Abstract: {item['abstract']}\n\n"
        f"Keywords: {item['keywords']}"
    )

    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0,
        max_output_tokens=1,
        top_p=0.1,
    )

    # IC1
    response = client.models.generate_content(
        model=modelGemini,
        contents="Using a 1-7 Likert scale (1 - Strongly disagree, 2 - Disagree, 3 - Somewhat disagree, "
                 "4 - Neither agree nor disagree, 5 - Somewhat agree, 6 - Agree, and 7 - Strongly agree) "
                 "rate your agreement with the question (only number): "
                 "<IC1 - Description of inclusion criteria 1.>",
        config=config,
    )
    r = response.text.strip()
    result_np[i, 1] = r[0]
    result_np[i, 6] = response.usage_metadata.total_token_count

    # IC2
    response = client.models.generate_content(
        model=modelGemini,
        contents="Using a 1-7 Likert scale (1 - Strongly disagree, 2 - Disagree, 3 - Somewhat disagree, "
                 "4 - Neither agree nor disagree, 5 - Somewhat agree, 6 - Agree, and 7 - Strongly agree) "
                 "rate your agreement with the question (only number): "
                 "<IC2 - Description of inclusion criteria n.>.",
        config=config,
    )
    r = response.text.strip()
    result_np[i, 2] = r[0]
    result_np[i, 7] = response.usage_metadata.total_token_count
    result_np[i, 8] = int(result_np[i, 6]) + int(result_np[i, 7])

    endTime = time.perf_counter()
    result_np[i, 3] = int(result_np[i, 1]) + int(result_np[i, 2])
    result_np[i, 4] = "I" if int(result_np[i, 3]) > 12 else "E"
    result_np[i, 5] = endTime - startTime

    print(f"Processing #{i+1} of {len(data)} in {round(result_np[i,5], 2)}s  Status: {result_np[i,4]}")
    i += 1

df = pd.DataFrame(result_np, columns=["Title", "IC1", "IC2", "SUM", "Status", "Execution time (s)", "Tokens IC1", "Tokens IC2", "Total Tokens"])

# Download file
xlsFile = "results-" + datetime.now().strftime('%Y-%m-%d-%H-%M-%S-') + modelGemini + ".xlsx"
df.to_excel(xlsFile, index=False, engine='openpyxl')
print(f"File '{xlsFile}' created!")
files.download(xlsFile)
display(df)

## 5. Press ⌘/Ctrl+F9 to run. The results will be automatically downloaded